# 01 — What Are Embeddings?

**Phase 1 · Notebook 1 of 5**

---

## Learning objectives

1. Explain what an embedding is in plain language
2. Generate sentence embeddings locally using `sentence-transformers`
3. Measure semantic similarity between pieces of text using cosine similarity
4. Build a simple nearest-neighbour search over a real dataset
5. Visualise an embedding space with PCA

---

## The core idea

An **embedding** maps text into a vector so that similar meanings end up close together in space.

**Tools used:** `sentence-transformers`, `numpy`, `pandas`, `scikit-learn`, `matplotlib`  
**Data:** `data/raw/articles.csv` (200 rows)

---
## 1 · Imports and setup

**What this cell does:** Imports every library we need and prints their versions.

**Why it matters:** Pinning versions is the first step in reproducible ML work.

**What to look for:** All imports succeed. If `sentence-transformers` is missing, run `pip install -r requirements.txt`.

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import sentence_transformers

print(f'Python               : {sys.version.split()[0]}')
print(f'numpy                : {np.__version__}')
print(f'pandas               : {pd.__version__}')
print(f'sentence-transformers: {sentence_transformers.__version__}')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
DATA_PATH    = os.path.join(PROJECT_ROOT, 'data', 'raw', 'articles.csv')
print(f'\nData file exists: {os.path.exists(DATA_PATH)}')

---
## 2 · Load the dataset

**What this cell does:** Reads `articles.csv` into a DataFrame and prints a summary.

**Why it matters:** Always inspect data before embedding — length distribution, balance, and nulls all affect quality.

**What to look for:** 200 rows, no nulls in `body`, roughly 40 rows per category.

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'\nNull counts:\n{df.isnull().sum()}')
print(f'\nCategory distribution:\n{df.category.value_counts()}')
df['body_len'] = df['body'].str.len()
print(f"\nBody length (chars): {df['body_len'].agg(['min','mean','max']).round(1).to_dict()}")
df.head(3)

---
## 3 · Load the embedding model

**What this cell does:** Loads `all-MiniLM-L6-v2` via sentence-transformers (downloads ~90 MB on first run).

**Why it matters:** Model choice is one of the highest-leverage decisions in any AI project. This model produces 384-dim vectors and runs on CPU.

**What to look for:** Vector shape `(384,)`, dtype `float32`.

In [ ]:
MODEL_NAME = 'all-MiniLM-L6-v2'
model = SentenceTransformer(MODEL_NAME)
test_vec = model.encode('Hello, embeddings!')
print(f'Model  : {MODEL_NAME}')
print(f'Shape  : {test_vec.shape}')
print(f'Dtype  : {test_vec.dtype}')
print(f'First 5: {test_vec[:5].round(4)}')

---
## 4 · Geometry of meaning — micro-demo

**What this cell does:** Encodes 6 sentences from 3 domains and plots pairwise cosine similarity.

**Why it matters:** Shows that similar meanings produce numerically close vectors — the foundation of RAG and semantic search.

**What to look for:** Same-domain pairs score > 0.8; cross-domain pairs score < 0.3; diagonal = 1.0.

In [ ]:
demo_sentences = [
    'Neural networks learn patterns from large amounts of data.',
    'Deep learning models discover features automatically from training examples.',
    'Chop the onions finely before adding them to the pan.',
    'Dice the vegetables and saute them over medium heat.',
    'The Andromeda galaxy is 2.5 million light-years from Earth.',
    'Black holes are regions where gravity is so strong nothing can escape.',
]
labels = ['ML-1','ML-2','Cook-1','Cook-2','Astro-1','Astro-2']
vecs = model.encode(demo_sentences)
sim  = cosine_similarity(vecs)

fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(sim, vmin=0, vmax=1, cmap='RdYlGn')
plt.colorbar(im, ax=ax, label='Cosine similarity')
ax.set_xticks(range(6)); ax.set_yticks(range(6))
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_yticklabels(labels)
ax.set_title('Pairwise cosine similarity — 6 demo sentences')
for i in range(6):
    for j in range(6):
        ax.text(j,i,f'{sim[i,j]:.2f}',ha='center',va='center',fontsize=9)
plt.tight_layout(); plt.show()
print(f'ML-1 vs ML-2   : {sim[0,1]:.4f}  (HIGH expected)')
print(f'ML-1 vs Cook-1 : {sim[0,2]:.4f}  (LOW expected)')

---
## 5 · Embed all 200 articles

**What this cell does:** Batch-encodes every article body; stores result as a (200, 384) NumPy array.

**Why it matters:** This is the offline indexing step — done once. The matrix is what gets stored in a vector DB.

**What to look for:** Shape `(200, 384)`, all norms ≈ 1.0, memory ~0.3 MB.

In [ ]:
article_vectors = model.encode(
    df['body'].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)
norms = np.linalg.norm(article_vectors, axis=1)
print(f'Shape : {article_vectors.shape}')
print(f'Norms : min={norms.min():.6f}  max={norms.max():.6f}  (should be ~1.0)')
print(f'Memory: {article_vectors.nbytes/1e6:.2f} MB')

---
## 6 · Semantic search

**What this cell does:** Encodes a query, scores it against all 200 article vectors, returns top-k.

**Why it matters:** This is the query-time half of RAG. The query shares meaning, not keywords, with the documents.

**What to look for:** Top results thematically match the query; scores typically 0.4–0.8.

In [ ]:
def semantic_search(query, vectors, dataframe, top_k=5):
    q_vec  = model.encode([query], normalize_embeddings=True)
    scores = cosine_similarity(q_vec, vectors)[0]
    idx    = np.argsort(scores)[::-1][:top_k]
    res    = dataframe.iloc[idx][['title','category','body']].copy()
    res['similarity'] = scores[idx].round(4)
    return res.reset_index(drop=True)

QUERY = 'advances in machine learning and artificial intelligence'
print(f'Query: "{QUERY}"\n')
pd.set_option('display.max_colwidth', 80)
display(semantic_search(QUERY, article_vectors, df))

---
## 7 · Try your own queries

**What this cell does:** Runs search for three cross-category queries.

**Why it matters:** Experience language-agnostic semantic matching firsthand.

**What to look for:** Returned categories match query intent; score < 0.25 means no good match exists.

In [ ]:
for q in ['climate crisis and renewable energy','mental health and stress management','cryptocurrency investment risks']:
    print(f'\nQuery: {q}')
    for _, row in semantic_search(q, article_vectors, df, top_k=3).iterrows():
        print(f"  [{row['similarity']:.3f}] [{row['category']:12s}] {row['title'][:65]}")

---
## 8 · Visualise with PCA

**What this cell does:** Reduces 384-dim vectors to 2D with PCA and plots by category.

**Why it matters:** If categories cluster in 2D, the model encodes topic structure — essential for search accuracy.

**What to look for:** Rough category clusters; overlapping health/science is expected and correct.

In [ ]:
pca = PCA(n_components=2, random_state=42)
xy  = pca.fit_transform(article_vectors)
exp = pca.explained_variance_ratio_ * 100
print(f'Variance explained: PC1={exp[0]:.1f}%  PC2={exp[1]:.1f}%  Total={exp.sum():.1f}%')

cats = df['category'].unique()
cmap = dict(zip(cats, cm.tab10.colors[:len(cats)]))
fig, ax = plt.subplots(figsize=(10,7))
for cat in cats:
    m = df['category']==cat
    ax.scatter(xy[m,0], xy[m,1], c=[cmap[cat]], label=cat, alpha=0.7, s=60, edgecolors='white', linewidths=0.4)
ax.set_xlabel(f'PC1 ({exp[0]:.1f}% var)'); ax.set_ylabel(f'PC2 ({exp[1]:.1f}% var)')
ax.set_title('Article embeddings — PCA 2D projection (coloured by category)')
ax.legend(title='Category'); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

---
## 9 · Per-category similarity heatmap

**What this cell does:** Computes average intra- and inter-category cosine similarity as a 5x5 matrix.

**Why it matters:** Shows how separable categories are — high diagonal, low off-diagonal = good retrieval precision.

**What to look for:** Diagonal highest in each row; health/science overlap is expected.

In [ ]:
cats = sorted(df['category'].unique())
n = len(cats)
avg = np.zeros((n,n))
for i,ci in enumerate(cats):
    for j,cj in enumerate(cats):
        avg[i,j] = cosine_similarity(article_vectors[df['category']==ci], article_vectors[df['category']==cj]).mean()
fig,ax = plt.subplots(figsize=(7,6))
im = ax.imshow(avg, vmin=0, vmax=1, cmap='Blues')
plt.colorbar(im, ax=ax, label='Mean cosine similarity')
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(cats, rotation=30, ha='right'); ax.set_yticklabels(cats)
ax.set_title('Average cosine similarity between article categories')
for i in range(n):
    for j in range(n):
        ax.text(j,i,f'{avg[i,j]:.3f}',ha='center',va='center',fontsize=10,
                color='white' if avg[i,j]>0.6 else 'black')
plt.tight_layout(); plt.show()

---
## 10 · Summary and key takeaways

**What this cell does:** Prints a recap and runs sanity assertions.

**Why it matters:** Assertions make notebooks self-verifying and catch regressions early.

**What to look for:** All assertions pass; takeaways recap what was learned.

In [ ]:
assert article_vectors.shape == (200, 384), 'Unexpected embedding shape'
assert np.allclose(np.linalg.norm(article_vectors, axis=1), 1.0, atol=1e-5), 'Vectors not normalised'
print('All assertions passed')
print('''
KEY TAKEAWAYS
1. An embedding converts text into a fixed-length numerical vector.
2. Semantically similar texts produce numerically close vectors.
3. Cosine similarity is the standard metric for normalised vectors.
4. Indexing (embed docs) is offline; search (embed query) is online.
5. PCA confirms category structure is preserved in embedding space.

NEXT: 02_similarity_and_bm25.ipynb — dense vs BM25 keyword search
''')